# tinyLMTune — Question Answering

This notebook demonstrates 3 ways to train TinyBERT for **qna** using tinyLMTune:

1. **Synthetic data** — auto-generated via Flan-T5/Mistral
2. **Benchmark data** — real HuggingFace dataset (SQuAD)
3. **Raw user data** — your own text, structured or unstructured

Each example runs the full pipeline: data → token analysis → search space recommendation → GA optimisation → model save → inference.

## Setup

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

# Install if needed (uncomment):
!pip install -e ../../tinylmtune_v2/

from tinylmtune import optimize_slm, TinyInference, print_token_analysis, print_recommendation

---
## Example 1 — Synthetic Data (via Flan-T5)

No data needed. Flan-T5/Mistral generates training data from a topic prompt.

**Requirements:** Flan-T5 must be installed and running (`pip install sentencepiece`), with `mistral` model pulled (``).

In [ ]:
best = optimize_slm(
    task="qna",
    corpus_prompt="Generate question-answer-context samples about world history and geography",
    n_examples=1000,
    max_len=8,
    pop_size=4,
    generations=2,
    output_dir="models/qna_synthetic",
)
print("Best config:", best)

### Inference on synthetic model

In [ ]:
model = TinyInference("models/qna_synthetic")
result = model.predict("What is the capital of France?", context="Paris is capital of France")
print(result)

---
## Example 2 — Benchmark Data (SQuAD)

Uses a real HuggingFace dataset. No Flan-T5 needed.

### Load SQuAD dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("squad", split="train")
ds = ds.shuffle(seed=42).select(range(5000))
benchmark_data = []
for r in ds:
    answer = r["answers"]["text"][0] if r["answers"]["text"] else ""
    if not answer:
        continue
    benchmark_data.append({
        "question": r["question"],
        "context": r["context"],
        "answer": answer,
    })
print(f"Loaded {len(benchmark_data)} records")
print(f"Sample keys: {benchmark_data[0].keys()}")
print(f"Sample Q: {benchmark_data[0]['question']}")
print(f"Sample A: {benchmark_data[0]['answer']}")
print(f"Sample C: {benchmark_data[0]['context'][:100]}...")

In [ ]:
print(benchmark_data[0].keys())
print(benchmark_data[0])

### Analyze token lengths

In [ ]:
print_token_analysis(benchmark_data, task="qna")

### Check recommended search space

In [ ]:
print_recommendation(n_samples=len(benchmark_data), task="qna")

In [ ]:
## Use recommended search space 

GA_SEARCH_SPACE = {'learning_rate': (5e-06, 0.001),
 'batch_size': [4, 8, 16, 32],
 'epochs': (2, 8),
 'warmup_ratio': (0.0, 0.3),
 'weight_decay': (0.0, 0.05),
 'dropout': (0.0, 0.2),
 'attention_dropout': (0.0, 0.2),
 'gradient_accumulation_steps': [1, 2, 4, 8],
 'lr_scheduler_type': ['linear',
  'cosine',
  'cosine_with_restarts',
  'constant_with_warmup'],
 'label_smoothing': (0.0, 0.1),
 'max_grad_norm': (0.5, 5.0)}

### Train with GA optimisation

In [ ]:
best = optimize_slm(
    task="qna",
    user_data=benchmark_data,
    max_len=32,
    pop_size=4,
    generations=2,
    output_dir="models/qna_benchmark",
)
print("Best config:", best)

### Visualize GA Results

5 plots showing how the GA searched for the best hyperparameters:
1. **Fitness progress** — best/avg/worst per generation
2. **Parameter scatter** — each param vs fitness (best = red star)
3. **Scheduler comparison** — box plot by LR scheduler type
4. **Config evolution** — how the best config changed over generations
5. **Population heatmap** — all individuals in the last generation

In [ ]:
from tinylmtune import plot_results, print_best_config_table

# Print formatted best config
print_best_config_table(best)

# Generate all 5 plots
figs = plot_results(best, save_dir="plots/benchmark")

### Inference on benchmark model

In [ ]:
model = TinyInference("models/qna_benchmark")
result = model.predict("Who invented the telephone?", context="Alexander Graham Bell invented the telephone in 1876.")
print(result)

---
## Example 3 — Raw User Data

Three sub-examples showing different input formats:
- **3a.** Structured dicts (correct format)
- **3b.** Raw text strings (auto-labelled via Flan-T5)
- **3c.** Wrong-format dicts (auto-detected and converted)

### 3a. Structured dicts (used directly, no Flan-T5)

In [ ]:
my_data = [
    {"question": "What is Python?", "answer": "A high-level programming language"},
    {"question": "Who created Linux?", "answer": "Linus Torvalds"},
    {"question": "What is TinyBERT?", "answer": "A distilled version of BERT"},
    {"question": "What does GA stand for?", "answer": "Genetic Algorithm"},
    {"question": "What is fine-tuning?", "answer": "Adapting a pretrained model to a specific task"},
    {"question": "What is tokenization?", "answer": "Splitting text into tokens for model input"},
    {"question": "What is NLP?", "answer": "Natural Language Processing"},
    {"question": "What is a transformer?", "answer": "A neural network architecture using attention"},
    {"question": "What is BERT?", "answer": "Bidirectional Encoder Representations from Transformers"},
    {"question": "What is overfitting?", "answer": "When a model memorises training data instead of learning patterns"},
    {"question": "What is dropout?", "answer": "Randomly deactivating neurons during training to prevent overfitting"},
    {"question": "What is a loss function?", "answer": "A function that measures how wrong the model predictions are"},
]

best = optimize_slm(
    task="qna",
    user_data=my_data,
    pop_size=4,
    generations=2,
    output_dir="models/qna_user",
)

### 3b. Raw text strings (requires Flan-T5)

In [ ]:
# Raw text — Flan-T5 generates Q&A pairs from it
raw_texts = [
    "The Eiffel Tower was built in 1889 for the World Fair in Paris.",
    "Water boils at 100 degrees Celsius at sea level.",
    "The human body has 206 bones in the adult skeleton.",
    "DNA stands for deoxyribonucleic acid and carries genetic information.",
    "The speed of light is approximately 300,000 kilometers per second.",
    "Mount Everest is the tallest mountain at 8,849 meters above sea level.",
]

best = optimize_slm(
    task="qna",
    user_data=raw_texts,
    pop_size=4,
    generations=1,
    output_dir="models/qna_raw",
)

### 3c. Wrong-format dicts (requires Flan-T5)

In [ ]:
# Dicts with non-standard keys
wrong_format = [
    {"query": "What year was the moon landing?", "response": "1969"},
    {"query": "Who painted the Mona Lisa?", "response": "Leonardo da Vinci"},
    {"query": "What is the largest ocean?", "response": "Pacific Ocean"},
]

best = optimize_slm(
    task="qna",
    user_data=wrong_format,
    pop_size=4,
    generations=1,
    output_dir="models/qna_wrong",
)

### Visualize user data results

In [ ]:
# Plot results from structured data training (Example 3a)
from tinylmtune import plot_results, print_best_config_table
print_best_config_table(best)
figs = plot_results(best, save_dir="plots/user_data")

### Inference

In [ ]:
model = TinyInference("models/qna_user")
result = model.predict("What is dropout?", context="Dropout randomly deactivates neurons during training.")
print(result)

---
## Summary

| Example | Data source | Flan-T5 needed | Best for |
|---------|-------------|---------------|----------|
| Synthetic | Auto-generated | Yes | Quick prototyping |
| Benchmark | SQuAD | No | Reproducible evaluation |
| User data | Your own text | Depends on format | Production use |

The GA searches 11 hyperparameters: `learning_rate`, `batch_size`, `epochs`, `warmup_ratio`, `weight_decay`, `dropout`, `attention_dropout`, `gradient_accumulation_steps`, `lr_scheduler_type`, `label_smoothing`, `max_grad_norm`.

`max_len` is automatically determined from your data's token length distribution (p95 percentile).